# Deer v3 detector comparison on Modal

I finished evaluating four models on Modal: YOLO11s, YOLO26s, GFL and D-FINE-L. I abandoned the remaining experiments because they would take too much compute. This closes the training comparison.

## Modal validation and test results

I evaluated four models on the same frozen Deer v3 splits with our shared evaluator. I set each model’s confidence cutoff on validation and kept it fixed for test. Precision, recall and F1 use IoU 0.50. Count MAE is the average absolute count error per frame.

### Validation · 358 frames, 1,436 labeled deer boxes

| Model | AP50 | AP50–95 | Precision | Recall | F1 | Confidence | Count MAE |
|---|---:|---:|---:|---:|---:|---:|---:|
| YOLO11s | 70.9% | 42.7% | 93.2% | 67.4% | 78.2% | 0.40 | 1.29 |
| YOLO26s | 73.3% | 43.9% | 95.0% | 68.1% | 79.3% | 0.31 | 1.25 |
| GFL | 76.1% | 44.6% | 92.6% | 70.4% | 80.0% | 0.51 | 1.12 |
| D-FINE-L | 81.7% | 58.6% | 91.8% | 76.7% | 83.5% | 0.45 | 1.00 |

### Held-out test · 238 frames, 575 labeled deer boxes

| Model | AP50 | AP50–95 | Precision | Recall | F1 | Confidence | Count MAE |
|---|---:|---:|---:|---:|---:|---:|---:|
| YOLO11s | 60.7% | 29.5% | 80.3% | 43.8% | 56.7% | 0.40 | 1.24 |
| YOLO26s | 70.3% | 33.6% | 71.2% | 66.3% | 68.6% | 0.31 | 0.87 |
| GFL | 69.5% | 31.4% | 71.8% | 67.3% | 69.5% | 0.51 | 0.82 |
| D-FINE-L | 65.6% | 33.1% | 42.4% | 74.6% | 54.1% | 0.45 | 2.11 |

YOLO26s had the highest test AP. GFL was slightly ahead on F1 and count error. D-FINE-L matched 429 boxes versus 381 for YOLO26s, but left 583 unmatched predictions versus 154. Those extra boxes are a lot for an operator to review. I’d start flight trials with YOLO26s. These are frame-level results on a mixed-source test set. Jetson performance and counts of distinct deer per flight are still unmeasured.


## Where the models missed on Modal

Each match is a labeled box on one frame, at IoU 0.50 and the model’s validation-selected confidence. One deer can appear in multiple frames.

| Model | BAMBI matches / 73 | Decoy matches / 14 | Rucervus matches / 393 | Rucervus unmatched boxes |
|---|---:|---:|---:|---:|
| YOLO11s | 19 | 10 | 146 | 37 |
| YOLO26s | 24 | 11 | 268 | 135 |
| GFL | 20 | 11 | 285 | 133 |
| D-FINE-L | 26 | 10 | 310 | 521 |

- **YOLO11s missed the most labels.** It missed 323 of 575 test boxes. It left the fewest unmatched predictions (62) and had the lowest recall (43.8%).
- **Dry grass was hard for all four.** YOLO26s and GFL both missed every labeled box on BAMBI frames `227_2223` and `227_5871`. Of YOLO26s’s 49 BAMBI misses, 14 had a matching box below the cutoff, five had a nearby box with too little overlap, and 30 had no nearby saved box.
- **Marsh scenes produced extra boxes and misses.** YOLO26s and GFL each missed all eight labels on the upside-down, oblique Rucervus frame `UAVSurveyDeer_mp4-0032`. D-FINE-L left 521 of its 583 unmatched boxes on Rucervus. These boxes can include duplicates, poorly placed detections, or unlabeled animals. Review the frame and labels before treating a detection as a background error.
- **Decoys still need operator review.** Both YOLO26s and GFL missed Mississippi slide 103. On slide 179, YOLO26s’s two unmatched boxes at its standard cutoff were a near-match and a duplicate. Its coyote candidate scored 0.014. The small coyote spot-check gives limited evidence about confusion with other decoys.

## Local YOLO26s crop test

I ran the saved Modal checkpoint on my Mac. Each of the four overlapping crops covered 60% of the frame width and height and used the same 1280 input setting as the full frame. I mapped detections back to the full image and merged overlapping boxes.

| Mode at confidence 0.31 | BAMBI matches | Decoy matches | Total recall | Unmatched boxes | Median inference |
|---|---:|---:|---:|---:|---:|
| Full frame | 24 / 73 | 11 / 14 | 35 / 87 (40%) | 10 | 72 ms |
| Four overlapping crops | 34 / 73 | 12 / 14 | 46 / 87 (53%) | 26 | 286 ms |

Crops matched 11 more boxes across 84 frames. Precision fell from 78% to 64%, and median inference took about four times as long. Combining full-frame and crop predictions matched the same 46 boxes and raised the unmatched count to 30. This was an exploratory recall comparison on a subset I’d already inspected. The timings are from the Mac.

## How I’d improve the flight workflow

- Use the saved YOLO26s checkpoint for full-frame inference. Let the operator mark missed deer and dismiss duplicate boxes. Try crops as a second pass on ambiguous frames and measure whether the extra detections justify the time and review.
- Evaluate on fresh dry-grass and decoy flights, including verified empty passes. Review labels at full resolution where box placement affects the match. Pick the confidence cutoff on separate footage before scoring a held-out flight.
- Measure distinct deer found per flight, extra boxes the operator reviews, and latency on the Jetson.

## What went into v3

I picked imagery for the downward camera we expect to fly over an open field in Medicine Hat. The decoys are our closest match. I tightened oversized boxes and kept frames with coyote or bird decoys as negatives when the original key confirmed there were no deer. Young Deer gives us good aerial views, mostly in green grass. BAMBI adds some brown open ground, AWIR adds RGB pen views, and Sherbrooke adds crowded summer scenes.

I reviewed 1,029 Rucervus candidates and kept 454 aerial views, including dry vegetation and greener field edges. We used image matching and capture information to keep related views together where we could. BAMBI is grouped by location, AWIR by date, and Young Deer and Rucervus use inferred scene groups.

| Source | Train | Validation | Test |
|---|---:|---:|---:|
| Young Deer | 1,570 | 48 | 60 |
| BAMBI | 831 | 44 | 56 |
| Mississippi decoys | 139 | 31 | 28 |
| AWIR RGB | 34 | 6 | 13 |
| Sherbrooke summer | 328 | 147 | 0 |
| Rucervus aerial | 291 | 82 | 81 |
| **Total images** | **3,193** | **358** | **238** |
| Deer boxes | 9,848 | 1,436 | 575 |
| Images with no deer boxes | 99 | 103 | 20 |

We still have limited dry yellow grass at competition-like angles, and our decoys come from one field. The test split includes 28 Mississippi decoy frames with 14 labeled deer boxes.

## Runs and checkpoints

All four used a nominal input size of 1280. The resize and training recipes differed. YOLO11s continued from the earlier laptop checkpoint, and its reported validation and test scores came from the Modal run.

| Model | What I ran | Evaluated checkpoint |
|---|---|---|
| YOLO11s | Anant’s baseline architecture on Deer v3 | `/yolo11s/fit/weights/best.pt` |
| YOLO26s | My smaller detector candidate for Jetson | `/yolo26s/fit/weights/best.pt` |
| GFL | My ResNeXt-101 detector with deformable convolutions | `/gfl_long/best_bbox_mAP_epoch_25.pth` |
| D-FINE-L | My transformer-based detector candidate | `/dfine_l/fit/checkpoint0019.pth` |

The experiment record has the Modal volume names, checkpoint hashes, and locations for the plots, predictions and inference profiles. [Experiment record](../../docs/experiments/summaries/2026-11-10-deer-v3-modal-comparison.md)